# Sourcing Desk: step-by-step walkthrough

Run the cells top to bottom. Each step is the same function the pipeline calls, and every call is also
recorded to `runs/<run_id>/<step>__<part>.json` (input, output, API calls, timing, errors).

- Crustdata responses are cached in `.cache/crustdata/`, so re-running a cell costs **0 credits**.
  Set `use_cache=False` to force fresh data.
- Kernel: pick the repo's `.venv` (Python 3.13).

In [ ]:
import json
import pandas as pd
from desk import ROOT, pipeline, steps, view
from desk.crustdata import Crustdata
from desk.trace import Run, list_runs, load_run

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 30)

# electronics_bom.yaml = the founder's words only (Gemini reads it);
# electronics_bom_manual.yaml = hand-picked part numbers (no LLM); vague_request.yaml = hard case 9
req = pipeline.load_request(ROOT / "examples" / "electronics_bom.yaml")
packs = steps.load_packs()
cd = Crustdata(use_cache=True)       # False = always hit the API (costs credits)
run = Run("notebook")
run.record("00_request", req)
print(run.dir)
req

## Step 1: understand the specs (a gate)
Gemini reads the founder's text. If anything is unclear, **nothing goes to step 2**: all questions are asked at once,
then Gemini re-reads the whole request with the answers (round 2, no further questions allowed; anything still open
gets the most common choice, listed in assumptions). Part numbers it suggests aren't trusted until step 2 finds them.

Questions appear as input boxes. To skip the prompts, pre-fill `ANSWERS` by line_id.

In [ ]:
ANSWERS = {}   # e.g. {"capacitor_10uf": "16V or more, X5R", "usb_c_connector": "SMD USB 2.0 receptacle"}

def ask(q):
    return ANSWERS.get(q.line_id or "request") or input(f"[{q.line_id or 'request'}] {q.question}\n> ")

sr = pipeline.clarify_specs(run, req, packs, ask)
pack, specs = packs[sr.category], sr.specs
print(f"{sr.parsed_by}: {sr.category}, {sr.units} units -> {sr.deliver_to} by {sr.deadline}, "
      f"data budget ₹{sr.data_budget_inr}")
view.specs(sr)

In [ ]:
# What was asked and answered (round 1 -> round 2), and what the model assumed
clarify = next((r["output"] for r in run.records if r["step"] == "01_clarify"), None)
display(pd.DataFrame({"Q and A": clarify["answers"]}) if clarify else "no questions: round 1 was clear")
pd.DataFrame({"assumption": sr.assumptions})

## Step 2: find suppliers, one part
`PART` is a line_id from the table above.

In [ ]:
PART = specs[0].line_id
spec = next(s for s in specs if s.line_id == PART)
find = run.step("02_find", steps.find_suppliers, spec, pack, cd, part=PART)
print(find.status.value, "| tried:", find.tried, "\n", find.reason)
view.quotes(find)

Price tiers behind each quote (the pipeline picks the cheapest tier we qualify for at our quantity).

In [ ]:
view.tiers(find)

Other parts seen on the way: different part numbers, or the same part number from another maker (possible clones). Substitute candidates later.

In [ ]:
view.quotes(find, alternatives=True)

Unknown sellers from the open search. Step 3 will check these; `domain` is empty when the link is a Google redirect.

In [ ]:
view.leads(find)

The API calls this step made, with credits and cache hits.

In [ ]:
view.calls(run.records[-1:])

### Drill into a raw page
The exact page text a quote was read from.

In [ ]:
q = next((q for q in find.quotes if q.cache_key and q.status.value == "FOUND"), None)
if q:
    print(q.vendor, q.url, "| cache_key:", q.cache_key)
    print(view.page_text(q.cache_key, chars=1500, around="Stock"))

## Steps 2 to 6 for the whole BOM
Step 4 (buy data) is still a stub; everything else is real (payments are mocked).

In [ ]:
finds = {s.line_id: run.step("02_find", steps.find_suppliers, s, pack, cd, part=s.line_id) for s in specs}
pd.DataFrame([{"part": k, "status": f.status.value, "reason": f.reason} for k, f in finds.items()])

### Step 2b: investigate unknown sellers (agent)
For each part, a bounded agent (Gemini tool calling: fetch, read page, Crustdata identify/enrich, search) works out
what an open-search lead really is. Prices come from the structured page read, red flags become quote flags, and
code still decides. Each agent run's transcript is below.

In [ ]:
from desk.investigate import investigate_leads
from desk.schemas import Status
invs = {}
for s in specs:
    invs[s.line_id] = inv = run.step("02b_investigate", investigate_leads, s, finds[s.line_id], pack, cd, part=s.line_id)
    finds[s.line_id].quotes += [q for q in inv.quotes if q.status != Status.NO_MATCH]
    finds[s.line_id].alternatives += [q for q in inv.quotes if q.status == Status.NO_MATCH]
pd.DataFrame([{"part": k, "stop": a.stop_reason, "turns": a.turns, "credits": a.credits, "llm_calls": a.llm_calls,
               **{x: (a.report or {}).get(x) for x in ("seller_name", "seller_domain", "seller_type", "is_seller",
                                                         "sells_exact_part", "ships_to_india", "red_flags")}}
              for k, inv in invs.items() for a in inv.agents])

In [ ]:
# One agent run, step by step: what it called, with what, and what came back
view.transcript(invs[PART].agents[0]) if invs[PART].agents else "no lead investigated for this part"

### Step 3: is each seller a real, stable company?
Crustdata Identify resolves the seller (via `company_domain` when the storefront resolves to something else, e.g.
digikey.in → DigiKey's Indian back office, so we check digikey.com). Enrich gives headcount, growth, HQ and traffic.
✓ passed, ✗ failed, ? couldn't check. Score = weighted share passed (weights and bars in the pack).

In [ ]:
checks = run.step("03_check", steps.check_suppliers, list(finds.values()), pack, cd)
view.checks(checks)

### Step 3b: RFQs (agent)
Sellers we found but couldn't price get a request for quote, best seller check first, only when a part has no
usable quote. The RFQ agent writes, follows up once, and records the quote (numbers must appear in the vendor's
email). A teammate plays the vendor: `python -m desk rfq reply <thread_id> "..."`, then re-run.

In [ ]:
from desk import rfq
rfq_out = run.step("03b_rfq", pipeline.rfq_step, pipeline.request_key(req), specs, list(finds.values()), checks,
                   pack, sr.deliver_to, sr.deadline)
pd.DataFrame(rfq_out) if rfq_out else "no RFQs needed: every part has a usable quote"

In [ ]:
gaps = run.step("04_buy_data", steps.buy_data, list(finds.values()), sr.data_budget_inr or 0)
pd.DataFrame(gaps)

### Step 5: decide
Rules in order: real company → ships to India in time → reliability bar ("cheaper but untrusted") → unflagged
(flagged and unverified go to the founder) → lowest landed cost (goods + shipping + duty + GST, rates in the pack).

In [ ]:
decisions = {s.line_id: run.step("05_decide", steps.decide, s, finds[s.line_id], checks, pack, sr.deadline,
                                 part=s.line_id) for s in specs}
display(pd.DataFrame([{"part": k, "status": d.status, "picked": d.picked.vendor if d.picked else None,
                       "landed_inr": d.total_inr, "per_unit": d.landed.per_unit_inr if d.landed else None,
                       "arrives_by": d.landed.arrives_by if d.landed else None, "reason": d.reason}
                      for k, d in decisions.items()]))
# Flagged options (possible clones, unverified sellers) waiting for the founder
pd.DataFrame([{"part": k, **f} for k, d in decisions.items() for f in d.founder_review])

In [ ]:
# Every vendor considered for one part, with landed cost and seller score, and the ones ruled out
d = decisions[PART]
display(pd.DataFrame(d.considered))
pd.DataFrame(d.rejected)

### Step 6: order and pay, exactly once
PLANNED → APPROVED → PAY_PENDING → PAID, saved to `desk.db` at every step. Orders above `approval_limit_inr` are held.
The key depends only on what's bought, so re-running this cell never pays twice. Approve a held order with
`orders.approve(key)` (or `python -m desk approve <key>`) and re-run the cell.

In [ ]:
rkey = pipeline.request_key(req)
orders_out = {k: run.step("06_order", steps.place_order, d, rkey, req.approval_limit_inr, part=k)
              for k, d in decisions.items()}
view.orders(rkey)

In [ ]:
from desk import orders
# What the (mock) vendor checkout actually charged: one row per order, however often you re-run
pd.DataFrame(orders.checkout_payments())

In [ ]:
# The agent's wallet (mock): balance, per-payment limit, and every rupee in and out
from desk import payments
w = payments.AgentWallet()
print(f"provider: {payments.provider().name} | balance ₹{w.balance:,.2f} | per-payment limit ₹{w.per_payment_limit:,.0f}")
display(pd.DataFrame(w.ledger()))
pd.DataFrame(payments.provider().events())   # what a real gateway would send as webhooks

## This run, step by step

In [ ]:
view.steps(run.records)

## Look at any past run
Includes runs from the CLI (`uv run python -m desk run examples/electronics_bom.yaml`).

In [ ]:
[p.name for p in list_runs()[:10]]

In [ ]:
records = load_run()                 # latest run; or load_run("20261122-131235_electronics_bom")
view.steps(records)

In [ ]:
rec = next(r for r in records if r["step"] == "02_find")
print(rec["part"])
print(json.dumps(rec["output"], indent=1, ensure_ascii=False)[:4000])